[![AnalyticsDojo](https://github.com/rpi-techfundamentals/fall2018-materials/blob/master/fig/final-logo.png?raw=1)](http://rpi.analyticsdojo.com)
<center><h1>Pytorch with the MNIST Dataset - MINST</h1></center>
<center><h3><a href = 'http://rpi.analyticsdojo.com'>rpi.analyticsdojo.com</a></h3></center>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rpi-techfundamentals/fall2018-materials/blob/master/10-deep-learning/04-pytorch-mnist.ipynb)



From Kaggle:
"MNIST ("Modified National Institute of Standards and Technology") is the de facto “hello world” dataset of computer vision. Since its release in 1999, this classic dataset of handwritten images has served as the basis for benchmarking classification algorithms. As new machine learning techniques emerge, MNIST remains a reliable resource for researchers and learners alike."

[Read more.](https://www.kaggle.com/c/digit-recognizer)


<a title="By Josef Steppan [CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0)], from Wikimedia Commons" href="https://commons.wikimedia.org/wiki/File:MnistExamples.png"><img width="512" alt="MnistExamples" src="https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png"/></a>

This code is adopted from the pytorch examples repository.
It is licensed under BSD 3-Clause "New" or "Revised" License.
Source: https://github.com/pytorch/examples/
LICENSE: https://github.com/pytorch/examples/blob/master/LICENSE

![](https://github.com/rpi-techfundamentals/fall2018-materials/blob/master/10-deep-learning/mnist-comparison.png?raw=1)
Table from [Wikipedia](https://en.wikipedia.org/wiki/MNIST_database)

In [1]:
!pip install torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

### Pytorch Advantages vs Tensorflow
- Pytorch Enables dynamic computational graphs (which change be changed) while Tensorflow is static.
- Tensorflow enables easier deployment.

In [35]:
#Import Libraries

from __future__ import print_function
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
from torch.utils.data import DataLoader

# Дефиниция на устройството: GPU ако е наличен, иначе CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [43]:
args={}
kwargs={}
args['batch_size']=512
args['test_batch_size']=1000
args['epochs']=35  #The number of Epochs is the number of times you go through the full dataset.
args['lr']=0.015 #Learning rate is how fast it will decend.
args['momentum']=0.5 #SGD momentum (default: 0.5) Momentum is a moving average of our gradients (helps to keep direction).
args['weight_decay']=5e-4
args['droput']=0.15

args['seed']=1 #random seed
args['log_interval']=10
args['cuda']=False


In [44]:
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    # Приложение на лека аффинна трансформация:
    # - degrees: завъртане до ±15°,
    # - translate: случайно изместване до 10% от размера по хоризонтала и вертикала,
    # - scale: леко увеличение или намаляване (90% - 110%)
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),

    # Алтернативно (ако предпочиташ по-проста трансформация):
    # transforms.RandomRotation(10),
    # transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),

    transforms.ToTensor(),
    # Нормализиране с предварително изчислените стойности за MNIST
    transforms.Normalize((0.1307,), (0.3081,))
])

# За test набора обикновено не прилагаме аугментация, само нормализация
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


In [45]:
# Зареждане на MNIST датасета, използвайки вече дефинираните трансформации
train_loader = DataLoader(
    datasets.MNIST('../data', train=True, download=True, transform=transform_train),
    batch_size=args['batch_size'], shuffle=True, **kwargs
)

test_loader = DataLoader(
    datasets.MNIST('../data', train=False, transform=transform_test),
    batch_size=args['test_batch_size'], shuffle=True, **kwargs
)

In [46]:
class Net(nn.Module):
    def __init__(self, dropout=0.3):
        super(Net, self).__init__()

        # Свивка 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.batchnorm1 = nn.BatchNorm2d(32)

        # Свивка 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.batchnorm2 = nn.BatchNorm2d(64)

        # Свивка 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)  # Намалява размера наполовина
        self.dropout = nn.Dropout(dropout)  # Dropout само за Fully Connected слоя

        # Напълно свързани слоеве
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.batchnorm1(self.conv1(x))))
        x = self.pool(F.relu(self.batchnorm2(self.conv2(x))))
        x = self.pool(F.relu(self.batchnorm3(self.conv3(x))))

        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(F.relu(self.fc1(x)))  # Dropout след активация
        x = self.fc2(x)
        return x

In [47]:
# Инициализиране на модела, loss функцията, оптимизатора и learning rate scheduler-а
net = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=args['lr'], weight_decay=args['weight_decay'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [48]:

def train(epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        if args['cuda']:
            data, target = data.cuda(), target.cuda()
        #Variables in Pytorch are differenciable.
        data, target = Variable(data), Variable(target)
        #This will zero out the gradients for this batch.
        optimizer.zero_grad()
        output = model(data)
        # Calculate the loss The negative log likelihood loss. It is useful to train a classification problem with C classes.
        loss = F.nll_loss(output, target)
        #dloss/dx for every Variable
        loss.backward()
        #to do a one-step update on our parameter.
        optimizer.step()
        #Print out the loss periodically.
        if batch_idx % args['log_interval'] == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

def test():
    model.eval()
    test_loss = 0
    correct = 0
    for data, target in test_loader:
        if args['cuda']:
            data, target = data.cuda(), target.cuda()
        data, target = Variable(data, volatile=True), Variable(target)
        output = model(data)
        test_loss += F.nll_loss(output, target, size_average=False).item() # sum up batch loss
        pred = output.data.max(1, keepdim=True)[1] # get the index of the max log-probability
        correct += pred.eq(target.data.view_as(pred)).long().cpu().sum()

    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))




In [49]:

# Тренировъчен цикъл
for epoch in range(args['epochs']):
    net.train()
    running_loss = 0.0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (batch_idx + 1) % 100 == 0:
            print(f"Train Epoch: {epoch+1} [{(batch_idx+1)*len(inputs)}/{len(train_loader.dataset)}] Loss: {loss.item():.4f}")
    scheduler.step()  # Обновяване на learning rate-а

    # Тестова фаза след всяка епоха
    net.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    test_loss /= len(test_loader)
    accuracy = 100. * correct / total
    print(f"Test set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{total} ({accuracy:.2f}%)")

print("Finished Training")

Train Epoch: 1 [51200/60000] Loss: 0.2472
Test set: Average loss: 0.2192, Accuracy: 9294/10000 (92.94%)
Train Epoch: 2 [51200/60000] Loss: 0.2075
Test set: Average loss: 0.0764, Accuracy: 9744/10000 (97.44%)
Train Epoch: 3 [51200/60000] Loss: 0.1494
Test set: Average loss: 0.0434, Accuracy: 9858/10000 (98.58%)
Train Epoch: 4 [51200/60000] Loss: 0.1617
Test set: Average loss: 0.0359, Accuracy: 9877/10000 (98.77%)
Train Epoch: 5 [51200/60000] Loss: 0.0805
Test set: Average loss: 0.0470, Accuracy: 9835/10000 (98.35%)
Train Epoch: 6 [51200/60000] Loss: 0.1158
Test set: Average loss: 0.0672, Accuracy: 9778/10000 (97.78%)
Train Epoch: 7 [51200/60000] Loss: 0.0678
Test set: Average loss: 0.0426, Accuracy: 9864/10000 (98.64%)
Train Epoch: 8 [51200/60000] Loss: 0.1327
Test set: Average loss: 0.0471, Accuracy: 9848/10000 (98.48%)
Train Epoch: 9 [51200/60000] Loss: 0.1057
Test set: Average loss: 0.0471, Accuracy: 9852/10000 (98.52%)
Train Epoch: 10 [51200/60000] Loss: 0.1458
Test set: Average los